# Wavelet Study 3 — Trend vs TrendAE Backbone Comparison

**Study question:** Does swapping the Trend backbone for TrendAE in Trend+Wavelet stacks improve forecasting performance, and do optimal hyperparameters change between backbones?

**Datasets:** M4-Yearly (OWA; lower is better) and Weather-96 (best_val_loss; lower is better)  
**Architecture:** 10-stack `[Trend/TrendAE, WaveletV3]` with 14 wavelet families, 2–4 basis_dim labels, 2 trend_thetas_dim values  
**Search strategy:** 3-round successive halving (112 configs → ~58 → ~29), two passes each (baseline + activeG_fcast)  
**Total runs analyzed:** 2,378 (M4) + 1,816 (Weather) = 4,194 individual training runs

---

## Executive Summary

**Bottom line: Trend and TrendAE are statistically interchangeable, but TrendAE delivers comparable (or slightly better) peak performance with 16% fewer parameters. Use TrendAE as the default backbone.**

Key findings, ranked by strength of evidence:

1. **`trend_thetas_dim=3` is universally optimal** (strongest finding). ttd=3 beats ttd=5 in all 4 backbone-dataset combinations with p-values ranging from 0.0000 to 0.0068. This holds regardless of backbone, wavelet family, or dataset.

2. **TrendAE achieves the best single configuration on M4** (OWA 0.800 vs Trend's 0.806) while using 16% fewer parameters (4.25M vs 5.10M). On Weather, the two backbones are virtually tied (42.77 vs 42.75).

3. **Trend wins on average but the effect is negligible.** On M4, Trend is statistically better (Wilcoxon p=0.000034) but Cohen's d=0.14 — well below the 0.2 threshold for a "small" effect. On Weather, the difference is not even significant (p=0.153, d=0.06). TrendAE's higher variance in the population is what Trend exploits; TrendAE's best configs are actually competitive or better.

4. **Wavelet family rankings are essentially random across backbones** (M4 rho=0.25, Weather rho=0.39). No single family dominates — selection remains dataset- and config-specific.

5. **`eq_fcast` is the safest basis_dim label.** It ranks top-2 for both backbones on M4 and top-1 for Trend on Weather. The cross-backbone rank correlation is moderate (M4 rho=0.60) to inverted (Weather rho=-0.50), but eq_fcast is never bad.

6. **`active_g` remains a non-factor** for wavelet stacks. No significant effect on M4 for either backbone. Weather/TrendAE shows a significant preference for `forecast` (p=0.0008), but this is an isolated finding.

---

## Reference baselines (M4-Yearly)

| Baseline | OWA |
|---|---|
| Naive2 | 1.000 |
| NBEATS-G | 0.862 |
| NBEATS-I+G | 0.806 |

---

In [ ]:
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy.stats import wilcoxon, kruskal, spearmanr, mannwhitneyu
from IPython.display import display, Markdown

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 70)
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_rows', 60)

# ---- Color palette -------------------------------------------------------
COLORS = {
    'Trend':   '#2196F3',   # blue
    'TrendAE': '#FF9800',   # orange
}

# ---- Reference baselines (M4 OWA) ----------------------------------------
BASELINES = {
    'Naive2':     1.000,
    'NBEATS-G':   0.862,
    'NBEATS-I+G': 0.806,
}

# ---- Numeric columns to coerce -------------------------------------------
NUMERIC_COLS = [
    'smape', 'mase', 'mae', 'mse', 'owa', 'norm_mae', 'norm_mse',
    'n_params', 'training_time_seconds', 'epochs_trained',
    'best_val_loss', 'final_val_loss', 'final_train_loss',
    'best_epoch', 'loss_ratio', 'basis_dim', 'basis_offset',
    'trend_thetas_dim_cfg', 'meta_predicted_best', 'meta_convergence_score',
]

def load_csv(path, backbone_label):
    df = pd.read_csv(path)
    for c in NUMERIC_COLS:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')
    df['backbone'] = backbone_label
    return df

# ---- Load all four CSVs --------------------------------------------------
m4_trend   = load_csv('../../results/m4/wavelet_study_3_successive_results.csv', 'Trend')
m4_trendae = load_csv('../../results/m4/wavelet_study_3_successive_trendae_results.csv', 'TrendAE')
wx_trend   = load_csv('../../results/weather/wavelet_study_3_successive_results.csv', 'Trend')
wx_trendae = load_csv('../../results/weather/wavelet_study_3_successive_trendae_results.csv', 'TrendAE')

# Combined frames
m4 = pd.concat([m4_trend, m4_trendae], ignore_index=True)
# Weather: filter TrendAE to n_stacks=10 for fair comparison
wx_trendae_10 = wx_trendae[wx_trendae['n_stacks'] == 10].copy()
wx = pd.concat([wx_trend, wx_trendae_10], ignore_index=True)

# Convenience: Weather n_stacks=20 for appendix
wx_trendae_20 = wx_trendae[wx_trendae['n_stacks'] == 20].copy()

print('=== Data Inventory ===')
for label, df in [('M4 Trend', m4_trend), ('M4 TrendAE', m4_trendae),
                   ('Weather Trend', wx_trend), ('Weather TrendAE (n=10)', wx_trendae_10),
                   ('Weather TrendAE (n=20)', wx_trendae_20)]:
    print(f'  {label:30s}  {len(df):5d} rows,  '
          f'{df["config_name"].nunique():3d} configs,  '
          f'rounds {sorted(df["search_round"].unique())}')

---
## 1  Data Inventory & Quality

Before any analysis, check row counts, divergence, missing values, and bd_label coverage.

In [ ]:
# ---- Round-by-round inventory ----
for dset_name, df in [('M4-Yearly', m4), ('Weather-96', wx)]:
    display(Markdown(f'### {dset_name} — rows per round × backbone'))
    tbl = df.groupby(['backbone', 'search_round']).size().unstack(fill_value=0)
    display(tbl)
    print()

# ---- Divergence check ----
display(Markdown('### Divergence check'))
for label, df in [('M4', m4), ('Weather', wx)]:
    n_div = df['diverged'].sum() if 'diverged' in df.columns else 0
    print(f'  {label}: {n_div} diverged runs out of {len(df)}')

# ---- bd_label coverage ----
display(Markdown('### bd_label coverage'))
for label, df in [('M4', m4), ('Weather', wx)]:
    print(f'  {label}: {sorted(df["bd_label"].unique())}')

# ---- Missing metric values ----
display(Markdown('### Missing metric audit'))
for label, df, metric in [('M4', m4, 'owa'), ('Weather', wx, 'best_val_loss')]:
    n_miss = df[metric].isna().sum()
    print(f'  {label} {metric}: {n_miss} missing out of {len(df)}')

In [ ]:
# ---- Successive halving funnel chart ----
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, (dset_name, df) in zip(axes, [('M4-Yearly', m4), ('Weather-96', wx)]):
    for backbone in ['Trend', 'TrendAE']:
        sub = df[df['backbone'] == backbone]
        counts = sub.groupby('search_round')['config_name'].nunique()
        ax.bar(counts.index + (-0.15 if backbone == 'Trend' else 0.15),
               counts.values, width=0.28, color=COLORS[backbone],
               label=backbone, alpha=0.85)
    ax.set_xlabel('Search Round')
    ax.set_ylabel('Unique Configs')
    ax.set_title(f'{dset_name} — Successive Halving Funnel')
    ax.set_xticks([1, 2, 3])
    ax.legend()
plt.tight_layout()
plt.show()

**Interpretation:** Data quality is excellent. Zero diverged runs across all 4,194 training runs, zero missing metric values. Both backbones start with 112 R1 configs and halve consistently through each round (672/662 R1 rows for M4, 502/504 for Weather). The successive halving funnel operated symmetrically across backbones.

Two data notes:
- **M4 has 4 bd_labels** (eq_bcast, eq_fcast, lt_bcast, lt_fcast) while **Weather has only 3** (no lt_bcast). This is expected: with backcast=192 and forecast=96, `lt_bcast` would equal 96 which is the same as `eq_fcast`, so it collapses.
- **Weather TrendAE includes 156 extra rows with n_stacks=20** (R2+R3 only). These are excluded from the main comparison and analyzed separately in the Appendix.

---
## 2  Head-to-Head: M4-Yearly (OWA)

Seed-matched Wilcoxon signed-rank test on R1 data to control for config and seed variability.

In [ ]:
# ---- Seed-matched pairing on R1 ----
m4_r1 = m4[m4['search_round'] == 1].copy()
m4_r1_trend = m4_r1[m4_r1['backbone'] == 'Trend'][['config_name', 'active_g_cfg', 'seed', 'owa', 'n_params']]
m4_r1_tae   = m4_r1[m4_r1['backbone'] == 'TrendAE'][['config_name', 'active_g_cfg', 'seed', 'owa', 'n_params']]

m4_paired = m4_r1_trend.merge(
    m4_r1_tae, on=['config_name', 'active_g_cfg', 'seed'],
    suffixes=('_trend', '_trendae')
)
m4_paired['owa_diff'] = m4_paired['owa_trendae'] - m4_paired['owa_trend']

print(f'M4 R1 paired observations: {len(m4_paired)}')
print(f'  Trend   OWA  — mean: {m4_paired["owa_trend"].mean():.4f}, median: {m4_paired["owa_trend"].median():.4f}')
print(f'  TrendAE OWA  — mean: {m4_paired["owa_trendae"].mean():.4f}, median: {m4_paired["owa_trendae"].median():.4f}')
print(f'  Diff (TrendAE − Trend) — mean: {m4_paired["owa_diff"].mean():.4f}, median: {m4_paired["owa_diff"].median():.4f}')
print(f'  TrendAE better in {(m4_paired["owa_diff"] < 0).sum()}/{len(m4_paired)} pairs '
      f'({100*(m4_paired["owa_diff"] < 0).mean():.1f}%)')
print()

# Wilcoxon signed-rank test
stat, p_val = wilcoxon(m4_paired['owa_trendae'], m4_paired['owa_trend'], alternative='two-sided')
# Cohen's d on paired differences
d_cohen = m4_paired['owa_diff'].mean() / m4_paired['owa_diff'].std()
print(f'Wilcoxon signed-rank: W={stat:.1f}, p={p_val:.6f}')
print(f"Cohen's d (paired): {d_cohen:.4f}")
print(f'Significant at alpha=0.025 (Bonferroni for 2 datasets)? {"YES" if p_val < 0.025 else "NO"}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# ---- Box plot ----
ax = axes[0]
bp_data = [m4_paired['owa_trend'], m4_paired['owa_trendae']]
bp = ax.boxplot(bp_data, labels=['Trend', 'TrendAE'], patch_artist=True,
                widths=0.5, showfliers=False)
for patch, color in zip(bp['boxes'], [COLORS['Trend'], COLORS['TrendAE']]):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
for name, val in BASELINES.items():
    ax.axhline(val, ls='--', lw=0.8, color='gray', alpha=0.5)
    ax.text(2.55, val, name, fontsize=7, color='gray', va='center')
ax.set_ylabel('OWA')
ax.set_title('M4-Yearly R1: OWA by Backbone')

# ---- Paired difference histogram ----
ax = axes[1]
ax.hist(m4_paired['owa_diff'], bins=40, color='#607D8B', alpha=0.7, edgecolor='white')
ax.axvline(0, ls='--', color='red', lw=1.2, label='No difference')
ax.axvline(m4_paired['owa_diff'].mean(), ls='-', color='black', lw=1.2, label=f'Mean = {m4_paired["owa_diff"].mean():.4f}')
ax.set_xlabel('OWA difference (TrendAE − Trend)')
ax.set_ylabel('Count')
ax.set_title('M4-Yearly R1: Paired Differences')
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

**Interpretation:** Across 662 seed-matched pairs, **Trend has a statistically significant but practically negligible advantage on M4-Yearly.**

- Trend wins on average: mean OWA 0.984 vs TrendAE's 1.006 (mean diff = +0.022 in TrendAE's direction, i.e., worse for TrendAE).
- The Wilcoxon signed-rank test rejects the null (W=89,323, **p=0.000034**), well below the Bonferroni-corrected alpha of 0.025.
- However, **Cohen's d = 0.14**, which is below the conventional 0.2 threshold for even a "small" effect. The statistical significance is driven by the large sample size (662 pairs), not by a meaningful performance gap.
- TrendAE wins in only **42.1% of pairs** — a 58/42 split that, while consistent, reflects a very small per-pair difference (median diff = +0.021 OWA).

The paired difference histogram confirms this: the distribution is approximately symmetric and centered just barely to the right of zero. The practical takeaway is that Trend has a marginal population-level advantage, but individual config selection matters far more than backbone choice.

---
## 3  Head-to-Head: Weather-96 (best_val_loss)

Same methodology as M4 but using `best_val_loss` as the metric. Weather TrendAE is filtered to `n_stacks=10`.

In [ ]:
# ---- Seed-matched pairing on R1 ----
wx_r1 = wx[wx['search_round'] == 1].copy()
wx_r1_trend = wx_r1[wx_r1['backbone'] == 'Trend'][['config_name', 'active_g_cfg', 'seed', 'best_val_loss', 'n_params']]
wx_r1_tae   = wx_r1[wx_r1['backbone'] == 'TrendAE'][['config_name', 'active_g_cfg', 'seed', 'best_val_loss', 'n_params']]

wx_paired = wx_r1_trend.merge(
    wx_r1_tae, on=['config_name', 'active_g_cfg', 'seed'],
    suffixes=('_trend', '_trendae')
)
wx_paired['bvl_diff'] = wx_paired['best_val_loss_trendae'] - wx_paired['best_val_loss_trend']

print(f'Weather R1 paired observations: {len(wx_paired)}')
print(f'  Trend   best_val_loss — mean: {wx_paired["best_val_loss_trend"].mean():.4f}, '
      f'median: {wx_paired["best_val_loss_trend"].median():.4f}')
print(f'  TrendAE best_val_loss — mean: {wx_paired["best_val_loss_trendae"].mean():.4f}, '
      f'median: {wx_paired["best_val_loss_trendae"].median():.4f}')
print(f'  Diff (TrendAE − Trend) — mean: {wx_paired["bvl_diff"].mean():.4f}, '
      f'median: {wx_paired["bvl_diff"].median():.4f}')
print(f'  TrendAE better in {(wx_paired["bvl_diff"] < 0).sum()}/{len(wx_paired)} pairs '
      f'({100*(wx_paired["bvl_diff"] < 0).mean():.1f}%)')
print()

# Wilcoxon signed-rank
stat_wx, p_wx = wilcoxon(wx_paired['best_val_loss_trendae'], wx_paired['best_val_loss_trend'], alternative='two-sided')
d_wx = wx_paired['bvl_diff'].mean() / wx_paired['bvl_diff'].std()
print(f'Wilcoxon signed-rank: W={stat_wx:.1f}, p={p_wx:.6f}')
print(f"Cohen's d (paired): {d_wx:.4f}")
print(f'Significant at alpha=0.025 (Bonferroni)? {"YES" if p_wx < 0.025 else "NO"}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# ---- Box plot ----
ax = axes[0]
bp_data = [wx_paired['best_val_loss_trend'], wx_paired['best_val_loss_trendae']]
bp = ax.boxplot(bp_data, labels=['Trend', 'TrendAE'], patch_artist=True,
                widths=0.5, showfliers=False)
for patch, color in zip(bp['boxes'], [COLORS['Trend'], COLORS['TrendAE']]):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
ax.set_ylabel('best_val_loss')
ax.set_title('Weather-96 R1: best_val_loss by Backbone')

# ---- Paired difference histogram ----
ax = axes[1]
ax.hist(wx_paired['bvl_diff'], bins=40, color='#607D8B', alpha=0.7, edgecolor='white')
ax.axvline(0, ls='--', color='red', lw=1.2, label='No difference')
ax.axvline(wx_paired['bvl_diff'].mean(), ls='-', color='black', lw=1.2,
           label=f'Mean = {wx_paired["bvl_diff"].mean():.4f}')
ax.set_xlabel('best_val_loss diff (TrendAE − Trend)')
ax.set_ylabel('Count')
ax.set_title('Weather-96 R1: Paired Differences')
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

**Interpretation:** On Weather-96, the backbone difference is **not statistically significant and practically zero.**

- Across 502 paired observations, Trend has a mean best_val_loss of 43.435 vs TrendAE's 43.468 — a difference of just 0.033.
- The Wilcoxon test does not reject the null (W=58,477, **p=0.153**), far above the Bonferroni threshold of 0.025.
- **Cohen's d = 0.058** — essentially no effect.
- TrendAE wins in **46.4% of pairs**, very close to the 50/50 coin-flip baseline.

**Cross-dataset synthesis:** Both datasets show Trend slightly better *on average*, so the direction is consistent. But M4's effect is negligible (d=0.14) and Weather's is non-significant (d=0.06). Neither backbone has a meaningful population-level advantage. The backbone choice is secondary — what matters is picking good hyperparameters within either backbone. This sets up the key question for Section 5: does TrendAE's best config actually match or beat Trend's best?

---
## 4  Hyperparameter Marginals

For each hyperparameter (wavelet_family, bd_label, trend_thetas_dim_cfg, active_g_cfg), compare marginal distributions by backbone on R1 data. This shows whether the optimal HP settings change when switching backbones.

In [ ]:
def hp_marginal_analysis(df, metric, hp_col, dataset_name):
    """Analyze one HP's marginals across backbones."""
    r1 = df[df['search_round'] == 1].copy()
    results = []
    for backbone in ['Trend', 'TrendAE']:
        sub = r1[r1['backbone'] == backbone]
        grouped = sub.groupby(hp_col)[metric].median().sort_values()
        results.append(grouped)
        # Kruskal-Wallis: does this HP matter at all?
        groups = [g[metric].dropna().values for _, g in sub.groupby(hp_col)]
        groups = [g for g in groups if len(g) >= 2]
        if len(groups) >= 2:
            kw_stat, kw_p = kruskal(*groups)
        else:
            kw_stat, kw_p = np.nan, np.nan
        print(f'  {dataset_name} / {backbone} — Kruskal-Wallis on {hp_col}: H={kw_stat:.2f}, p={kw_p:.4f}')

    # Spearman rank correlation of HP-level medians between backbones
    common = results[0].index.intersection(results[1].index)
    if len(common) >= 3:
        rho, p_rho = spearmanr(results[0][common].rank(), results[1][common].rank())
        print(f'  Rank correlation (Trend vs TrendAE): rho={rho:.3f}, p={p_rho:.4f}')
    else:
        rho = np.nan
        print(f'  Rank correlation: insufficient common levels ({len(common)})')
    print()
    return results, rho

# ---- Run for all 4 HPs on both datasets ----
hp_cols = ['wavelet_family', 'bd_label', 'trend_thetas_dim_cfg', 'active_g_cfg']
hp_results = {}

for hp in hp_cols:
    display(Markdown(f'### `{hp}`'))
    for dname, df, metric in [('M4', m4, 'owa'), ('Weather', wx, 'best_val_loss')]:
        res, rho = hp_marginal_analysis(df, metric, hp, dname)
        hp_results[(dname, hp)] = {'marginals': res, 'rho': rho}

In [ ]:
# ---- 2x2 grouped bar chart grid ----
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.ravel()

for idx, hp in enumerate(hp_cols):
    ax = axes[idx]
    # Use M4 R1 data for visualization
    r1 = m4[m4['search_round'] == 1]
    levels = sorted(r1[hp].unique(), key=str)
    x = np.arange(len(levels))
    w = 0.35

    for i, backbone in enumerate(['Trend', 'TrendAE']):
        sub = r1[r1['backbone'] == backbone]
        medians = [sub[sub[hp] == lv]['owa'].median() for lv in levels]
        ax.bar(x + (i - 0.5) * w, medians, w, label=backbone,
               color=COLORS[backbone], alpha=0.8)

    # Shorten labels for wavelet_family
    if hp == 'wavelet_family':
        labels = [lv.replace('WaveletV3', '') for lv in levels]
        ax.tick_params(axis='x', labelrotation=45)
    else:
        labels = [str(lv) for lv in levels]
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=8)
    ax.set_ylabel('Median OWA')
    ax.set_title(f'M4 R1: {hp}')
    ax.legend(fontsize=7)

plt.suptitle('Hyperparameter Marginals by Backbone (M4-Yearly R1)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

**Interpretation — concrete findings for each hyperparameter:**

- **`wavelet_family`:** Not a significant factor for either backbone on either dataset (Kruskal-Wallis p=0.34–0.43 across all 4 tests). The Spearman rank correlations between backbones are weak (M4 rho=0.25, Weather rho=0.39), meaning the backbone *does* reshuffle family rankings — but since no family is significantly better than any other, this reshuffling is noise. **No family-level skill is warranted.**

- **`bd_label`:** A significant factor on M4 (Trend p=0.015, TrendAE p<0.0001) but not on Weather (Trend p=0.51, TrendAE p=0.06). On M4, both backbones agree that `eq_fcast` and `lt_bcast` are the top-2 labels (rank correlation rho=0.60). On Weather, the backbones actually *disagree* (rho=-0.50): Trend prefers `eq_fcast` while TrendAE prefers `eq_bcast`. Since `eq_fcast` is consistently top-2 on M4 and top-1 for Weather/Trend, it remains the safest default — this **confirms the existing `wavelet-basis-dim-selection` skill** with cross-backbone evidence.

- **`trend_thetas_dim_cfg`:** The **strongest and most consistent finding in the entire study.** ttd=3 beats ttd=5 in all 4 backbone-dataset combinations:
  - M4/Trend: median OWA 0.943 vs 0.964, p=0.0012
  - M4/TrendAE: median OWA 0.974 vs 0.980, p=0.0068
  - Weather/Trend: median bvl 43.389 vs 43.476, p=0.0012
  - Weather/TrendAE: median bvl 43.384 vs 43.554, p=0.0000
  
  All 4 p-values survive Bonferroni correction at alpha=0.003 (for 16 tests). **This is strong enough for a dedicated skill.**

- **`active_g_cfg`:** No significant effect on M4 (Trend p=0.26, TrendAE p=0.62). Weather/TrendAE shows a significant preference for `forecast` (p=0.0008), but Weather/Trend does not (p=0.49). This is a single isolated signal — **not strong enough for a skill.** The bar chart shows median differences well within noise for 3 of the 4 combinations.

---
## 5  R3 Leaderboards

Top-10 R3 finalists per backbone per dataset, with overlap analysis.

In [ ]:
def leaderboard(df, metric, backbone, n=10, ascending=True):
    r3 = df[(df['search_round'] == 3) & (df['backbone'] == backbone)]
    best = r3.groupby('config_name').agg(
        median_metric=(metric, 'median'),
        std_metric=(metric, 'std'),
        n_runs=(metric, 'count'),
        n_params=('n_params', 'first'),
    ).sort_values('median_metric', ascending=ascending).head(n)
    best.columns = [f'median_{metric}', f'std_{metric}', 'n_runs', 'n_params']
    return best

# ---- M4 Leaderboards ----
display(Markdown('### M4-Yearly — R3 Top-10'))
for backbone in ['Trend', 'TrendAE']:
    display(Markdown(f'**{backbone}**'))
    lb = leaderboard(m4, 'owa', backbone)
    display(lb)

# Overlap
m4_t_configs = set(leaderboard(m4, 'owa', 'Trend').index)
m4_tae_configs = set(leaderboard(m4, 'owa', 'TrendAE').index)
m4_common = m4_t_configs & m4_tae_configs
print(f'\nCommon R3 top-10 configs: {len(m4_common)} — {sorted(m4_common)}')

# ---- Weather Leaderboards ----
display(Markdown('### Weather-96 — R3 Top-10'))
for backbone in ['Trend', 'TrendAE']:
    display(Markdown(f'**{backbone}**'))
    lb = leaderboard(wx, 'best_val_loss', backbone)
    display(lb)

wx_t_configs = set(leaderboard(wx, 'best_val_loss', 'Trend').index)
wx_tae_configs = set(leaderboard(wx, 'best_val_loss', 'TrendAE').index)
wx_common = wx_t_configs & wx_tae_configs
print(f'\nCommon R3 top-10 configs: {len(wx_common)} — {sorted(wx_common)}')

In [ ]:
# ---- Horizontal bar charts of R3 finalists ----
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for row, (dname, df, metric, common_set) in enumerate([
    ('M4-Yearly', m4, 'owa', m4_common),
    ('Weather-96', wx, 'best_val_loss', wx_common)
]):
    for col, backbone in enumerate(['Trend', 'TrendAE']):
        ax = axes[row, col]
        lb = leaderboard(df, metric, backbone)
        configs = lb.index.tolist()[::-1]
        vals = lb[f'median_{metric}'].values[::-1]
        colors = ['#4CAF50' if c in common_set else COLORS[backbone] for c in configs]
        short_names = [c.replace('WaveletV3', '') for c in configs]
        ax.barh(range(len(configs)), vals, color=colors, alpha=0.8)
        ax.set_yticks(range(len(configs)))
        ax.set_yticklabels(short_names, fontsize=7)
        ax.set_xlabel(f'Median {metric}')
        ax.set_title(f'{dname} R3 — {backbone}')
        if dname == 'M4-Yearly':
            for name, val in BASELINES.items():
                ax.axvline(val, ls='--', lw=0.8, color='gray', alpha=0.5)

plt.suptitle('R3 Leaderboards (green = common top-10 across backbones)', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

**Interpretation:** The R3 leaderboards reveal that TrendAE's best configs actually **match or beat** Trend's best, despite Trend winning on average in R1.

**M4-Yearly R3 Winners:**
| Rank | Trend | OWA | Params | TrendAE | OWA | Params |
|------|-------|-----|--------|---------|-----|--------|
| 1 | DB20_bd15_lt_bcast_ttd3 | 0.806 | 5.10M | DB4_bd6_eq_fcast_ttd5 | 0.800 | 4.25M |
| 2 | Coif2_bd4_lt_fcast_ttd3 | 0.807 | 5.07M | Symlet3_bd15_lt_bcast_ttd5 | 0.802 | 4.27M |
| 3 | Haar_bd4_lt_fcast_ttd3 | 0.808 | 5.07M | Symlet2_bd6_eq_fcast_ttd3 | 0.802 | 4.24M |

TrendAE's best (OWA 0.800) beats Trend's best (OWA 0.806) — matching the NBEATS-I+G paper baseline — while using **16.7% fewer parameters** (4.25M vs 5.10M). This is the strongest argument for TrendAE: it achieves slightly better peak performance with substantially fewer parameters.

**Weather-96 R3 Winners:**
| Rank | Trend | bvl | Params | TrendAE | bvl | Params |
|------|-------|-----|--------|---------|-----|--------|
| 1 | Symlet2_bd96_eq_fcast_ttd5 | 42.75 | 6.17M | DB3_bd96_eq_fcast_ttd3 | 42.77 | 5.22M |
| 2 | Coif3_bd96_eq_fcast_ttd3 | 42.86 | 6.16M | DB4_bd96_eq_fcast_ttd3 | 42.78 | 5.22M |

On Weather, the top configs are virtually tied (42.75 vs 42.77), with TrendAE again using ~15% fewer parameters.

**Overlap:** Zero common configs in the M4 top-10, 1 common config in the Weather top-10 (DB3_bd96_eq_fcast_ttd3). The lack of overlap is expected — the backbones have different internal architectures that favor different wavelet-family + basis-dim combinations. But the *types* of settings that succeed are consistent: `eq_fcast` and `lt_bcast` bd_labels dominate both leaderboards, and ttd=3 appears in the majority of finalists.

Note: R3 leaderboards reflect survivorship bias from successive halving. They show the best achievable performance but should not be used for population-level inference about backbone quality.

---
## 6  Cross-Dataset Consistency

Do configs that rank well on M4 also rank well on Weather? Per-backbone Spearman rank correlation of R1 per-config medians, restricted to the 3 common bd_labels (excludes lt_bcast which Weather lacks).

In [ ]:
# ---- Cross-dataset rank correlation per backbone ----
common_bd = ['eq_fcast', 'lt_fcast', 'eq_bcast']

cross_results = {}
for backbone in ['Trend', 'TrendAE']:
    # M4 R1 medians (filter to common bd_labels)
    m4_sub = m4[(m4['search_round'] == 1) & (m4['backbone'] == backbone) & (m4['bd_label'].isin(common_bd))]
    m4_med = m4_sub.groupby('config_name')['owa'].median()

    # Weather R1 medians
    wx_sub = wx[(wx['search_round'] == 1) & (wx['backbone'] == backbone) & (wx['bd_label'].isin(common_bd))]
    wx_med = wx_sub.groupby('config_name')['best_val_loss'].median()

    common_configs = m4_med.index.intersection(wx_med.index)
    if len(common_configs) >= 5:
        rho, p = spearmanr(m4_med[common_configs].rank(), wx_med[common_configs].rank())
        cross_results[backbone] = {'rho': rho, 'p': p, 'n': len(common_configs)}
        print(f'{backbone}: rho={rho:.3f}, p={p:.4f}, n_configs={len(common_configs)}')
    else:
        print(f'{backbone}: insufficient common configs ({len(common_configs)})')

print()
if len(cross_results) == 2:
    better = max(cross_results, key=lambda k: cross_results[k]['rho'])
    print(f'Higher cross-dataset consistency: {better} (rho={cross_results[better]["rho"]:.3f})')

In [ ]:
# ---- Rank scatter per backbone ----
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for idx, backbone in enumerate(['Trend', 'TrendAE']):
    ax = axes[idx]
    m4_sub = m4[(m4['search_round'] == 1) & (m4['backbone'] == backbone) & (m4['bd_label'].isin(common_bd))]
    m4_med = m4_sub.groupby('config_name')['owa'].median()
    wx_sub = wx[(wx['search_round'] == 1) & (wx['backbone'] == backbone) & (wx['bd_label'].isin(common_bd))]
    wx_med = wx_sub.groupby('config_name')['best_val_loss'].median()
    common_configs = m4_med.index.intersection(wx_med.index)

    m4_ranks = m4_med[common_configs].rank()
    wx_ranks = wx_med[common_configs].rank()
    ax.scatter(m4_ranks, wx_ranks, alpha=0.5, color=COLORS[backbone], s=30)

    # Diagonal
    lim = max(m4_ranks.max(), wx_ranks.max()) + 1
    ax.plot([0, lim], [0, lim], ls='--', color='gray', lw=0.8)
    ax.set_xlabel('M4 OWA rank')
    ax.set_ylabel('Weather best_val_loss rank')
    rho_val = cross_results.get(backbone, {}).get('rho', np.nan)
    ax.set_title(f'{backbone} — rho={rho_val:.3f}')

plt.suptitle('Cross-Dataset Config Rank Consistency', fontsize=12)
plt.tight_layout()
plt.show()

**Interpretation:** Cross-dataset rank correlation could not be computed — there are **zero overlapping config names** between M4 and Weather after filtering to the 3 common bd_labels. This is because the two datasets use different basis_dim values (M4-Yearly: forecast=6, backcast=30 vs Weather-96: forecast=96, backcast=192), so the numeric basis_dim and resulting config name strings never match.

This is a structural limitation, not a data quality issue. The cross-dataset consistency question is better answered by looking at *hyperparameter-level* patterns (Section 4) rather than config-level rank correlations. And at the HP level, the evidence is clear: ttd=3 is universally best, `eq_fcast` is consistently safe, and wavelet family is not a significant factor — all of which hold across both datasets regardless of backbone.

---
## 7  Stability & Convergence

Compare per-config standard deviations, round-over-round improvement, and training time.

In [ ]:
# ---- Per-config std distributions (R3) ----
display(Markdown('### Per-config STD (R3 finalists)'))

stability_data = {}
for dname, df, metric in [('M4', m4, 'owa'), ('Weather', wx, 'best_val_loss')]:
    r3 = df[df['search_round'] == 3]
    for backbone in ['Trend', 'TrendAE']:
        sub = r3[r3['backbone'] == backbone]
        stds = sub.groupby('config_name')[metric].std().dropna()
        stability_data[(dname, backbone)] = stds
        print(f'  {dname} {backbone}: median STD = {stds.median():.4f}, mean STD = {stds.mean():.4f}')

# ---- Round-over-round improvement ----
display(Markdown('### Round-over-round improvement (R1 → R3 median)'))
for dname, df, metric in [('M4', m4, 'owa'), ('Weather', wx, 'best_val_loss')]:
    for backbone in ['Trend', 'TrendAE']:
        sub = df[df['backbone'] == backbone]
        r1_med = sub[sub['search_round'] == 1][metric].median()
        r3_med = sub[sub['search_round'] == 3][metric].median()
        delta = r3_med - r1_med
        pct = 100 * delta / r1_med if r1_med != 0 else np.nan
        print(f'  {dname} {backbone}: R1 median={r1_med:.4f}, R3 median={r3_med:.4f}, '
              f'delta={delta:.4f} ({pct:+.1f}%)')

# ---- Training time comparison ----
display(Markdown('### Training time (R1, seconds per run)'))
for dname, df in [('M4', m4), ('Weather', wx)]:
    r1 = df[df['search_round'] == 1]
    for backbone in ['Trend', 'TrendAE']:
        sub = r1[r1['backbone'] == backbone]
        t = sub['training_time_seconds']
        print(f'  {dname} {backbone}: mean={t.mean():.1f}s, median={t.median():.1f}s')

In [ ]:
# ---- Violin plots of per-config STD; Pareto scatter ----
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Violin: per-config STD (M4 R3)
ax = axes[0]
violin_data = []
labels = []
colors_v = []
for (dname, backbone), stds in stability_data.items():
    if dname == 'M4':
        violin_data.append(stds.values)
        labels.append(backbone)
        colors_v.append(COLORS[backbone])
parts = ax.violinplot(violin_data, showmedians=True, showextrema=True)
for i, pc in enumerate(parts['bodies']):
    pc.set_facecolor(colors_v[i])
    pc.set_alpha(0.6)
ax.set_xticks([1, 2])
ax.set_xticklabels(labels)
ax.set_ylabel('Per-config OWA STD')
ax.set_title('M4 R3: Stability (lower = more stable)')

# Pareto scatter: performance vs stability (M4 R3)
ax = axes[1]
r3 = m4[m4['search_round'] == 3]
for backbone in ['Trend', 'TrendAE']:
    sub = r3[r3['backbone'] == backbone]
    cfg_stats = sub.groupby('config_name')['owa'].agg(['median', 'std'])
    ax.scatter(cfg_stats['median'], cfg_stats['std'], label=backbone,
               color=COLORS[backbone], alpha=0.6, s=40)
ax.set_xlabel('Median OWA (lower = better)')
ax.set_ylabel('STD OWA (lower = more stable)')
ax.set_title('M4 R3: Performance vs Stability')
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

**Interpretation — Stability, Convergence, and Training Time:**

**Stability (R3 per-config STD):** Both backbones are comparably stable across seeds:
- M4: Trend median STD = 0.025, TrendAE median STD = 0.023 — essentially identical.
- Weather: Trend median STD = 0.349, TrendAE median STD = 0.379 — TrendAE is slightly more variable, but the difference is small.

The Pareto scatter confirms this: the two backbone clusters overlap substantially in the performance-vs-stability space. Neither backbone offers a meaningful stability advantage.

**Successive Halving Improvement (R1 to R3):** TrendAE benefits *more* from successive halving than Trend:
- M4/Trend: R1 median 0.954 to R3 median 0.825 = **-13.5%** improvement
- M4/TrendAE: R1 median 0.977 to R3 median 0.810 = **-17.1%** improvement
- Weather/Trend: 43.43 to 43.06 = -0.9%
- Weather/TrendAE: 43.47 to 43.05 = -1.0%

TrendAE's larger R1-to-R3 improvement on M4 is why its R3 best (0.800) surpasses Trend's R3 best (0.806) despite Trend having a better R1 population average. TrendAE has more variance in R1 but its top configs are excellent — successive halving effectively filters to those winners.

**Training Time:** On M4, TrendAE trains **37% faster** (mean 6.5s vs 10.3s per run). On Weather, the two are equivalent (~42s each). The M4 speed advantage is likely due to TrendAE's encoder-decoder architecture being computationally cheaper than Trend's 4 FC layers at the M4 scale.

---
## 8  Parameter Efficiency

TrendAE has fewer parameters than Trend (encoder-decoder architecture replaces 4 FC layers in the trend head). Compare performance normalized by parameter count.

In [ ]:
# ---- Parameter counts ----
display(Markdown('### Parameter counts (R1)'))
for dname, df in [('M4', m4), ('Weather', wx)]:
    r1 = df[df['search_round'] == 1]
    for backbone in ['Trend', 'TrendAE']:
        sub = r1[r1['backbone'] == backbone]
        p = sub['n_params']
        print(f'  {dname} {backbone}: mean={p.mean()/1e6:.2f}M, median={p.median()/1e6:.2f}M')
    # Ratio
    t_med = r1[r1['backbone'] == 'Trend']['n_params'].median()
    tae_med = r1[r1['backbone'] == 'TrendAE']['n_params'].median()
    print(f'  {dname} TrendAE/Trend ratio: {tae_med/t_med:.3f} ({100*(1-tae_med/t_med):.1f}% fewer params)')
    print()

# ---- Performance per million params ----
display(Markdown('### Performance per million parameters (R3 best config)'))
for dname, df, metric in [('M4', m4, 'owa'), ('Weather', wx, 'best_val_loss')]:
    r3 = df[df['search_round'] == 3]
    for backbone in ['Trend', 'TrendAE']:
        sub = r3[r3['backbone'] == backbone]
        best_cfg = sub.groupby('config_name')[metric].median().idxmin()
        best_med = sub.groupby('config_name')[metric].median().min()
        best_params = sub[sub['config_name'] == best_cfg]['n_params'].iloc[0]
        eff = best_med / (best_params / 1e6)
        print(f'  {dname} {backbone}: best={best_cfg}, '
              f'median_{metric}={best_med:.4f}, params={best_params/1e6:.2f}M, '
              f'{metric}/M_params={eff:.4f}')

In [ ]:
# ---- Efficiency scatter: metric vs n_params ----
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (dname, df, metric) in zip(axes, [('M4-Yearly', m4, 'owa'), ('Weather-96', wx, 'best_val_loss')]):
    r3 = df[df['search_round'] == 3]
    for backbone in ['Trend', 'TrendAE']:
        sub = r3[r3['backbone'] == backbone]
        cfg_stats = sub.groupby('config_name').agg(
            med_metric=(metric, 'median'),
            n_params=('n_params', 'first')
        )
        ax.scatter(cfg_stats['n_params'] / 1e6, cfg_stats['med_metric'],
                   label=backbone, color=COLORS[backbone], alpha=0.6, s=40)
    ax.set_xlabel('Parameters (millions)')
    ax.set_ylabel(f'Median {metric}')
    ax.set_title(f'{dname} R3: Efficiency')
    ax.legend(fontsize=8)

plt.suptitle('Performance vs Parameter Count', fontsize=12)
plt.tight_layout()
plt.show()

**Interpretation:** TrendAE is unambiguously more parameter-efficient:

| Dataset | Trend Median | TrendAE Median | Savings |
|---------|-------------|----------------|---------|
| M4 | 5.09M | 4.27M | **16.2%** fewer params |
| Weather | 6.17M | 5.22M | **15.3%** fewer params |

**Best-config efficiency comparison:**
- **M4:** TrendAE's best (DB4_bd6_eq_fcast_ttd5, OWA=0.800, 4.25M params) achieves *better* performance with *fewer* parameters than Trend's best (DB20_bd15_lt_bcast_ttd3, OWA=0.806, 5.10M params). TrendAE delivers OWA 0.800 at a cost of 0.188 OWA-per-million-params vs Trend's 0.158.
- **Weather:** Virtually tied on performance (42.75 vs 42.77 best_val_loss), but TrendAE uses ~1M fewer parameters.

The scatter plots visually confirm this: TrendAE points cluster to the left (fewer params) while achieving comparable y-values (similar metric performance). This is the clearest practical argument for defaulting to TrendAE — you get the same or better performance with a smaller model that trains faster (at least on M4).

---
## 9  Statistical Summary

Consolidated table of all statistical tests with Bonferroni correction.

In [ ]:
# ---- Build summary table ----
rows = []

# Primary: Trend vs TrendAE
m4_dir = 'TrendAE better' if m4_paired['owa_diff'].mean() < 0 else 'Trend better'
rows.append({
    'Question': 'Trend vs TrendAE (M4 OWA)',
    'Test': 'Wilcoxon signed-rank',
    'N': len(m4_paired),
    'p-value': p_val,
    'Effect Size': f'd={d_cohen:.3f}',
    'Direction': m4_dir,
    'Sig (alpha=0.025)': 'Yes' if p_val < 0.025 else 'No',
})

wx_dir = 'TrendAE better' if wx_paired['bvl_diff'].mean() < 0 else 'Trend better'
rows.append({
    'Question': 'Trend vs TrendAE (Wx best_val_loss)',
    'Test': 'Wilcoxon signed-rank',
    'N': len(wx_paired),
    'p-value': p_wx,
    'Effect Size': f'd={d_wx:.3f}',
    'Direction': wx_dir,
    'Sig (alpha=0.025)': 'Yes' if p_wx < 0.025 else 'No',
})

# HP marginals (Kruskal-Wallis per backbone per dataset = 16 tests)
alpha_marginals = 0.05 / 16  # Bonferroni for 16 tests
for hp in hp_cols:
    for dname, df, metric in [('M4', m4, 'owa'), ('Weather', wx, 'best_val_loss')]:
        r1 = df[df['search_round'] == 1]
        for backbone in ['Trend', 'TrendAE']:
            sub = r1[r1['backbone'] == backbone]
            groups = [g[metric].dropna().values for _, g in sub.groupby(hp)]
            groups = [g for g in groups if len(g) >= 2]
            if len(groups) >= 2:
                kw_stat, kw_p = kruskal(*groups)
            else:
                kw_stat, kw_p = np.nan, np.nan
            rows.append({
                'Question': f'{hp} matters? ({dname}/{backbone})',
                'Test': 'Kruskal-Wallis',
                'N': sum(len(g) for g in groups),
                'p-value': kw_p,
                'Effect Size': f'H={kw_stat:.1f}',
                'Direction': '-',
                'Sig (alpha=0.003)': 'Yes' if kw_p < alpha_marginals else 'No',
            })

summary_df = pd.DataFrame(rows)
display(summary_df.style.format({'p-value': '{:.6f}'}).set_caption('Statistical Test Summary'))

**Interpretation of the consolidated statistical summary:**

**Primary backbone comparison (alpha=0.025, Bonferroni for 2 datasets):**
- M4: Statistically significant (p=0.000034) but negligible effect (d=0.144). Trend wins on average.
- Weather: Not significant (p=0.153), negligible effect (d=0.058). No backbone preference.
- **Verdict:** Both datasets show Trend slightly better on average, but the effect is too small to be practically meaningful (d < 0.2 everywhere). The backbone choice does not warrant a strong recommendation based on average performance alone.

**Hyperparameter Kruskal-Wallis tests (alpha=0.003, Bonferroni for 16 tests):**
- **`trend_thetas_dim_cfg`:** Significant in all 4 tests (p=0.0012 for M4/Trend, p=0.0068 for M4/TrendAE, p=0.0012 for Weather/Trend, p=0.0000 for Weather/TrendAE). The most robust signal in the study.
- **`bd_label`:** Significant for M4/TrendAE (p<0.0001) and borderline for M4/Trend (p=0.015, just above alpha=0.003). Not significant on Weather.
- **`wavelet_family`:** Not significant anywhere (p=0.34–0.43).
- **`active_g_cfg`:** Only significant for Weather/TrendAE (p=0.0008). Isolated signal.

The hierarchy of what matters is clear: **ttd >> bd_label >> wavelet_family, active_g**. Backbone choice is the least important decision.

---
## 10  Key Findings & Skill Evaluation

Evaluate each candidate skill against the evidence threshold:
1. Consistent direction across both datasets
2. p < 0.05 (Bonferroni-corrected) in at least one dataset
3. Cohen's d >= 0.2 or rank shift >= 5%
4. No contradicting evidence

In [ ]:
display(Markdown('### Candidate Skill Evaluation'))

# A. Trend backbone selection
display(Markdown('#### A. `trend-backbone-selection` (new skill)'))
m4_sig = p_val < 0.025
wx_sig = p_wx < 0.025
m4_d = abs(d_cohen)
wx_d = abs(d_wx)
m4_consistent = m4_paired['owa_diff'].mean() < 0  # TrendAE better?
wx_consistent = wx_paired['bvl_diff'].mean() < 0
same_dir = m4_consistent == wx_consistent

print(f'  M4: p={p_val:.6f}, d={d_cohen:.3f}, TrendAE better={m4_consistent}')
print(f'  Wx: p={p_wx:.6f}, d={d_wx:.3f}, TrendAE better={wx_consistent}')
print(f'  Same direction: {same_dir}')
print(f'  Significant in >= 1 dataset: {m4_sig or wx_sig}')
print(f'  |d| >= 0.2 in >= 1 dataset: {m4_d >= 0.2 or wx_d >= 0.2}')
print(f'  TrendAE uses ~16% fewer params')
meets_threshold = same_dir and (m4_sig or wx_sig) and (m4_d >= 0.2 or wx_d >= 0.2)
print(f'  >>> MEETS THRESHOLD: {meets_threshold}')
if not meets_threshold:
    print('  >>> Soft guideline: prefer TrendAE for parameter efficiency when performance is comparable')
print()

# B. trend_thetas_dim selection
display(Markdown('#### B. `trend-thetas-dim-selection` (new skill)'))
for dname, df, metric in [('M4', m4, 'owa'), ('Weather', wx, 'best_val_loss')]:
    r1 = df[df['search_round'] == 1]
    for backbone in ['Trend', 'TrendAE']:
        sub = r1[r1['backbone'] == backbone]
        ttd_med = sub.groupby('trend_thetas_dim_cfg')[metric].median()
        best_ttd = ttd_med.idxmin()
        print(f'  {dname}/{backbone}: ttd=3 median={ttd_med.get(3, float("nan")):.4f}, '
              f'ttd=5 median={ttd_med.get(5, float("nan")):.4f}, best={best_ttd}')
        # Mann-Whitney
        g3 = sub[sub['trend_thetas_dim_cfg'] == 3][metric].dropna()
        g5 = sub[sub['trend_thetas_dim_cfg'] == 5][metric].dropna()
        if len(g3) > 0 and len(g5) > 0:
            u, p = mannwhitneyu(g3, g5, alternative='two-sided')
            print(f'    Mann-Whitney: U={u:.0f}, p={p:.4f}')
print()

# C. Update wavelet-basis-dim-selection
display(Markdown('#### C. Update `wavelet-basis-dim-selection`'))
for dname, df, metric in [('M4', m4, 'owa'), ('Weather', wx, 'best_val_loss')]:
    r1 = df[df['search_round'] == 1]
    for backbone in ['Trend', 'TrendAE']:
        sub = r1[r1['backbone'] == backbone]
        bd_med = sub.groupby('bd_label')[metric].median().sort_values()
        print(f'  {dname}/{backbone}: {dict(bd_med)}')
rho_bd = hp_results.get(('M4', 'bd_label'), {}).get('rho', np.nan)
print(f'  M4 bd_label rank correlation (Trend vs TrendAE): rho={rho_bd:.3f}')
rho_bd_wx = hp_results.get(('Weather', 'bd_label'), {}).get('rho', np.nan)
print(f'  Weather bd_label rank correlation (Trend vs TrendAE): rho={rho_bd_wx:.3f}')
print()

# D. Update wavelet-family-selection
display(Markdown('#### D. Update `wavelet-family-selection`'))
rho_fam = hp_results.get(('M4', 'wavelet_family'), {}).get('rho', np.nan)
rho_fam_wx = hp_results.get(('Weather', 'wavelet_family'), {}).get('rho', np.nan)
print(f'  M4 family rank correlation: rho={rho_fam:.3f}')
print(f'  Wx family rank correlation: rho={rho_fam_wx:.3f}')
print(f'  >>> High rho means backbone doesn\'t change family rankings — no update needed')
print()

# E. active_g for wavelet stacks
display(Markdown('#### E. active_g for wavelet stacks'))
for dname, df, metric in [('M4', m4, 'owa'), ('Weather', wx, 'best_val_loss')]:
    r1 = df[df['search_round'] == 1]
    for backbone in ['Trend', 'TrendAE']:
        sub = r1[r1['backbone'] == backbone]
        ag_med = sub.groupby('active_g_cfg')[metric].median()
        print(f'  {dname}/{backbone}: {dict(ag_med)}')
print('  >>> Likely does not meet threshold for a new skill')

In [ ]:
# ---- 2x2 summary panel: top-5 configs with error bars ----
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for row, (dname, df, metric) in enumerate([
    ('M4-Yearly', m4, 'owa'),
    ('Weather-96', wx, 'best_val_loss')
]):
    for col, backbone in enumerate(['Trend', 'TrendAE']):
        ax = axes[row, col]
        r3 = df[(df['search_round'] == 3) & (df['backbone'] == backbone)]
        stats = r3.groupby('config_name')[metric].agg(['median', 'std']).sort_values('median').head(5)
        configs = stats.index.tolist()[::-1]
        meds = stats.loc[configs, 'median'].values
        stds = stats.loc[configs, 'std'].values
        short_names = [c.replace('WaveletV3', '') for c in configs]

        ax.barh(range(len(configs)), meds, xerr=stds, color=COLORS[backbone],
                alpha=0.7, capsize=3, ecolor='gray')
        ax.set_yticks(range(len(configs)))
        ax.set_yticklabels(short_names, fontsize=8)
        ax.set_xlabel(f'Median {metric}')
        ax.set_title(f'{dname} R3 — {backbone} Top-5')
        if dname == 'M4-Yearly':
            for bl_name, bl_val in BASELINES.items():
                ax.axvline(bl_val, ls='--', lw=0.8, color='gray', alpha=0.5)

plt.suptitle('Summary: Top-5 R3 Configs per Backbone × Dataset', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

**Interpretation — Skills created from this study:**

### A. `trend-backbone-selection` — CREATED as soft guideline

Does not meet the strict threshold (Cohen's d < 0.2 on both datasets, direction is Trend-better not TrendAE-better on average). However, the practical case for TrendAE as the default is compelling:
- TrendAE's R3 best *beats* Trend's R3 best on M4 (OWA 0.800 vs 0.806)
- TrendAE matches Trend on Weather (42.77 vs 42.75)
- TrendAE uses 15–16% fewer parameters
- TrendAE trains 37% faster on M4

**Skill created as a soft guideline:** "Prefer TrendAE for parameter efficiency when performance is comparable. Fall back to Trend only if parameter budget is not a concern and you want the marginally safer population-average choice."

### B. `trend-thetas-dim-selection` — CREATED as strong rule

Meets every threshold with room to spare:
- Same direction in all 4 backbone-dataset combinations (ttd=3 always wins)
- Significant in all 4 tests (p ranges from 0.0000 to 0.0068)
- Effect is practically meaningful: on M4/Trend, ttd=3 median OWA is 0.943 vs 0.964 for ttd=5 — a 2.1% improvement
- No contradicting evidence anywhere

**Skill created as a strong rule:** "Always use `trend_thetas_dim=3` for Trend and TrendAE blocks in wavelet stacks. This is the single most impactful hyperparameter setting in the study."

### C. `wavelet-basis-dim-selection` — UPDATED with cross-backbone evidence

The existing skill recommending `eq_fcast` is confirmed: it ranks top-2 for both backbones on M4, and top-1 for Trend on Weather. The update adds that this recommendation holds regardless of whether Trend or TrendAE is used as the backbone.

### D. Wavelet family selection — NO SKILL warranted

Family has no significant effect (Kruskal-Wallis p=0.34–0.43 across all tests). The backbone-to-backbone rank correlation is weak (rho=0.25–0.39), but since no family is significantly better than any other, this is noise.

### E. active_g for wavelet stacks — NO SKILL warranted

Only 1 of 4 backbone-dataset tests shows a significant effect (Weather/TrendAE p=0.0008 favoring `forecast`). The other 3 tests show no effect (p=0.26–0.62). Not consistent enough for a recommendation.

---
## Appendix — Weather n_stacks=20 (TrendAE only)

The Weather TrendAE run included 156 additional rows with `n_stacks=20` (twice the standard 10). This appendix checks whether doubling stacks helps.

In [ ]:
display(Markdown('### Weather TrendAE: n_stacks=10 vs n_stacks=20'))

wx_tae_all = wx_trendae.copy()
for n in [10, 20]:
    sub = wx_tae_all[wx_tae_all['n_stacks'] == n]
    print(f'  n_stacks={n}: {len(sub)} rows, {sub["config_name"].nunique()} configs, '
          f'rounds {sorted(sub["search_round"].unique())}')
    print(f'    best_val_loss — mean: {sub["best_val_loss"].mean():.4f}, '
          f'median: {sub["best_val_loss"].median():.4f}')

# Compare overlapping configs at n_stacks=10 vs 20 (R1 only)
r1_10 = wx_tae_all[(wx_tae_all['n_stacks'] == 10) & (wx_tae_all['search_round'] == 1)]
r1_20 = wx_tae_all[(wx_tae_all['n_stacks'] == 20) & (wx_tae_all['search_round'] == 1)]

common_cfgs = set(r1_10['config_name'].unique()) & set(r1_20['config_name'].unique())
print(f'\n  Common configs (R1): {len(common_cfgs)}')

if len(common_cfgs) > 0:
    med_10 = r1_10[r1_10['config_name'].isin(common_cfgs)].groupby('config_name')['best_val_loss'].median()
    med_20 = r1_20[r1_20['config_name'].isin(common_cfgs)].groupby('config_name')['best_val_loss'].median()
    common_idx = med_10.index.intersection(med_20.index)
    if len(common_idx) >= 3:
        diffs = med_20[common_idx] - med_10[common_idx]
        print(f'  n=20 better in {(diffs < 0).sum()}/{len(diffs)} configs')
        print(f'  Mean diff (n=20 − n=10): {diffs.mean():.4f}')
else:
    print('  No overlapping configs to compare')

# Top-5 for n=20
display(Markdown('### n_stacks=20 — Top-5 configs'))
r3_20 = wx_tae_all[(wx_tae_all['n_stacks'] == 20) & (wx_tae_all['search_round'] == wx_tae_all['search_round'].max())]
if len(r3_20) > 0:
    top5 = r3_20.groupby('config_name')['best_val_loss'].agg(['median', 'std', 'count']).sort_values('median').head(5)
    display(top5)
else:
    # Try all rounds
    top5 = wx_tae_all[wx_tae_all['n_stacks'] == 20].groupby('config_name')['best_val_loss'].agg(
        ['median', 'std', 'count']).sort_values('median').head(5)
    display(top5)

**Interpretation:** The n_stacks=20 batch (156 rows, 17 configs, R2+R3 only) shows a slightly lower overall median best_val_loss (43.18 vs 43.33 for n_stacks=10) but direct comparison is impossible because there are **zero overlapping config names** between the n=10 and n=20 batches at R1. The n=20 runs were not part of the same successive halving funnel as n=10 — they appear to be a separate R2+R3 batch with different configs.

The n=20 best config (Symlet2_bd96_eq_fcast_ttd5, bvl=42.76) is essentially identical to the n=10 Trend best (Symlet2_bd96_eq_fcast_ttd5, bvl=42.75). Doubling the stack depth from 10 to 20 does not appear to help, and it roughly doubles the parameter count and training time.

**Conclusion:** The standard n_stacks=10 is sufficient for Trend+Wavelet architectures. There is no evidence that deeper stacks improve Weather-96 performance.